In [14]:
import pandas as pd
import numpy as np
import re
from sklearn.model_selection import train_test_split

df = pd.read_csv(
    '/content/Reviews.csv',
    engine='python',      # forgiving parser
    on_bad_lines='skip',  # skip malformed rows
    quoting=3             # QUOTE_NONE
)


df = df[['Score', 'Text']]


df = df[df['Score'] != 3]
df['Sentiment'] = np.where(df['Score'] > 3, 1, 0)  # 1 = Positive, 0 = Negative


def clean_text(text):
    text = str(text).lower()                   # lowercase
    text = re.sub(r"[^a-z\s]", "", text)       # remove punctuation, numbers
    text = re.sub(r"\s+", " ", text).strip()   # remove extra whitespace
    return text

df['Text'] = df['Text'].apply(clean_text)


X_train, X_test, y_train, y_test = train_test_split(
    df['Text'], df['Sentiment'], test_size=0.2, random_state=42
)

print("Training set size:", X_train.shape[0])
print("Test set size:", X_test.shape[0])
print(X_train.head())

Training set size: 77302
Test set size: 19326
53090    i have been purchasing wild tribe moka frappe ...
89274    i found the flea trap to be helpful it does wo...
81492    excellent european taste dumpling soup for las...
57845    i personally dont care for the taste of coconu...
17821    my husband and i love these they taste soo goo...
Name: Text, dtype: object


In [15]:
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, classification_report

# TF-IDF vectorization
tfidf = TfidfVectorizer(max_features=5000)
X_train_tfidf = tfidf.fit_transform(X_train)
X_test_tfidf = tfidf.transform(X_test)

# Logistic Regression model
lr_model = LogisticRegression(max_iter=200)
lr_model.fit(X_train_tfidf, y_train)
y_pred_lr = lr_model.predict(X_test_tfidf)

print("TF-IDF + Logistic Regression Accuracy:", accuracy_score(y_test, y_pred_lr))
print(classification_report(y_test, y_pred_lr))

TF-IDF + Logistic Regression Accuracy: 0.9445306840525717
              precision    recall  f1-score   support

           0       0.89      0.68      0.77      2640
           1       0.95      0.99      0.97     16686

    accuracy                           0.94     19326
   macro avg       0.92      0.83      0.87     19326
weighted avg       0.94      0.94      0.94     19326



In [17]:
!pip install gensim


   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 27.9/27.9 MB 35.7 MB/s eta 0:00:00


In [20]:
import nltk

# Download the standard punkt tokenizer
nltk.download('punkt')

# Also download the tab version (sometimes required in Colab/Kaggle)
nltk.download('punkt_tab')

[nltk_data] Downloading package punkt to /root/nltk_data...
[nltk_data]   Package punkt is already up-to-date!
[nltk_data] Downloading package punkt_tab to /root/nltk_data...
[nltk_data]   Unzipping tokenizers/punkt_tab.zip.


True

In [21]:
from nltk.tokenize import word_tokenize

X_train_tokens = [word_tokenize(text.lower()) for text in X_train]
X_test_tokens = [word_tokenize(text.lower()) for text in X_test]

In [22]:
from nltk.tokenize import TreebankWordTokenizer

tokenizer = TreebankWordTokenizer()
X_train_tokens = [tokenizer.tokenize(text.lower()) for text in X_train]
X_test_tokens = [tokenizer.tokenize(text.lower()) for text in X_test]

In [23]:
from gensim.models import Word2Vec
from nltk.tokenize import word_tokenize
import nltk
nltk.download('punkt')

# Tokenize
X_train_tokens = [word_tokenize(text) for text in X_train]
X_test_tokens = [word_tokenize(text) for text in X_test]

# Train Word2Vec
w2v_model = Word2Vec(sentences=X_train_tokens, vector_size=100, window=5, min_count=2, workers=4)

# Create average embeddings for each review
def avg_word2vec(tokens, model, size):
    vec = np.zeros(size)
    count = 0
    for word in tokens:
        if word in model.wv:
            vec += model.wv[word]
            count += 1
    if count != 0:
        vec /= count
    return vec

X_train_w2v = np.array([avg_word2vec(tokens, w2v_model, 100) for tokens in X_train_tokens])
X_test_w2v = np.array([avg_word2vec(tokens, w2v_model, 100) for tokens in X_test_tokens])

from sklearn.ensemble import RandomForestClassifier
rf_model = RandomForestClassifier(n_estimators=100, random_state=42)
rf_model.fit(X_train_w2v, y_train)
y_pred_w2v = rf_model.predict(X_test_w2v)

print("Word2Vec + Random Forest Accuracy:", accuracy_score(y_test, y_pred_w2v))

[nltk_data] Downloading package punkt to /root/nltk_data...
[nltk_data]   Package punkt is already up-to-date!


Word2Vec + Random Forest Accuracy: 0.9302494049467039


In [19]:
from tensorflow.keras.preprocessing.text import Tokenizer
from tensorflow.keras.preprocessing.sequence import pad_sequences
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Embedding, LSTM, Dense, Dropout

# Tokenization and padding
tokenizer = Tokenizer(num_words=10000)
tokenizer.fit_on_texts(X_train)
X_train_seq = pad_sequences(tokenizer.texts_to_sequences(X_train), maxlen=200)
X_test_seq = pad_sequences(tokenizer.texts_to_sequences(X_test), maxlen=200)

# LSTM Model
model = Sequential([
    Embedding(input_dim=10000, output_dim=128, input_length=200),
    LSTM(128, dropout=0.2, recurrent_dropout=0.2),
    Dense(1, activation='sigmoid')
])
model.compile(loss='binary_crossentropy', optimizer='adam', metrics=['accuracy'])
model.fit(X_train_seq, y_train, epochs=3, batch_size=128, validation_split=0.1)

loss, acc = model.evaluate(X_test_seq, y_test)
print("RNN (LSTM) Accuracy:", acc)

/usr/local/lib/python3.12/dist-packages/keras/src/layers/core/embedding.py:100: UserWarning: Argument `input_length` is deprecated. Just remove it.
  warnings.warn(


Epoch 1/3
544/544 ━━━━━━━━━━━━━━━━━━━━ 738s 1s/step - accuracy: 0.9232 - loss: 0.2020 - val_accuracy: 0.9497 - val_loss: 0.1398
Epoch 2/3
544/544 ━━━━━━━━━━━━━━━━━━━━ 696s 1s/step - accuracy: 0.9578 - loss: 0.1174 - val_accuracy: 0.9524 - val_loss: 0.1413
Epoch 3/3
544/544 ━━━━━━━━━━━━━━━━━━━━ 690s 1s/step - accuracy: 0.9669 - loss: 0.0909 - val_accuracy: 0.9582 - val_loss: 0.1248
604/604 ━━━━━━━━━━━━━━━━━━━━ 48s 80ms/step - accuracy: 0.9540 - loss: 0.1295
RNN (LSTM) Accuracy: 0.9539998173713684


lstm gud because because of highest accuracy and captures semantic and contextual information unlike other models
